# Does the student repeat the teacher's mistake? TraSeR vs the SVG2 pipeline

**Teacher** = the SVG2 pipeline we ran (SAM2 → DAM → LLM → relations). On our trampoline clip it called the **floor (id 0) a "person"**, and that broke relations ("shoe on person(0)").

**Student** = TraSeR, the model the paper trained on 636K videos labelled by that teacher.

**Question:** given the *same* object masks, does TraSeR also call the floor a person?
- If **yes**: the teacher's errors are learned by the model (bad labels → bad model).
- If **no**: the trained model is more robust than the pipeline that made its data.

**Run 2:** the same, but with the floor mask pulled back from the boy (the fix that worked for DAM). Does TraSeR's answer change?

**Uses saved results only:** our stage-2 tracks + stage-3 cleanup, and the 40 objects DAM described. No SAM2. **Time:** model download (8 GB, ~5 min) + 2 runs × ~3–5 min on one T4.

In [ ]:
import os, sys, json, subprocess, time
# use the T4 with the most free memory (another open notebook may still hold a model on GPU 0)
free = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.split()
GPU = max(range(len(free)), key=lambda i: int(free[i])) if free else 0
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
print("free MiB per GPU:", free, "-> using GPU", GPU)
if free and int(free[GPU]) < 10000:
    print("!! less than 10 GB free: shut down other notebook kernels (they keep their models on the GPU), then rerun")
REPO = "/kaggle/working/svg2-implementation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", "claude/traser-implementation-guide-xlshit",
                    "https://github.com/kanu2704/svg2-implementation", REPO], check=True)
os.chdir(REPO)
os.environ["HF_HOME"] = "/tmp/hf_cache"
r = subprocess.run("bash setup_kaggle.sh", shell=True, capture_output=True, text=True)
print(r.stdout[-600:], r.stderr[-600:])
sys.path.insert(0, os.path.join(REPO, "third_party/SVG2/pipeline"))
import numpy as np, cv2
from pycocotools import mask as mask_utils
import svg2_pipeline as P
OUT, DATA = "notes/traser_exp", "data/traser_exp"
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
FLOOR, BOY_PARTS = 0, [1, 21, 38]

## 1. Inputs for TraSeR, from our saved results

- **Video:** the first 120 frames of `sav_000001.mp4` (the part we tracked), written as a short clip. TraSeR samples ~1 frame/s from it.
- **Masks:** our stage-3 objects, in TraSeR's input format: for every frame, a list with one mask (COCO RLE) per object, empty where the object is absent.
- **Which objects:** the same 40 that DAM described, so we can compare label by label.

In [ ]:
cfg = P.PipelineConfig()
tracks = json.load(open("data/stage12/sav_000001_first120_tracks.json"))
cleaned = P.stage3_cleanup(cfg, tracks)                       # same stage 3 as before
T, H, W = cleaned["total_frames"], cleaned["height"], cleaned["width"]
columns = [o["object_id"] for o in cleaned["objects"]]       # column k of the mask list = this object id
col_of = {oid: k for k, oid in enumerate(columns)}

# short clip = the frames our masks cover
VIDEO = f"{DATA}/sav_000001_first{T}.mp4"
cap = cv2.VideoCapture("third_party/SVG2/pipeline/examples/sav_000001.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
wr = cv2.VideoWriter(VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
for _ in range(T):
    ok, fr = cap.read()
    if not ok:
        break
    wr.write(fr)
wr.release(); cap.release()
print("clip:", VIDEO, int(cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FRAME_COUNT)), "frames at", round(fps, 2), "fps")

def write_masks(path, override=None):
    """TraSeR input: mask_data[frame][column] = RLE or None. override = {object_id: {frame: bool mask}}."""
    data = [[o["masks"][f] for o in cleaned["objects"]] for f in range(T)]
    for oid, per_frame in (override or {}).items():
        for f, m in per_frame.items():
            rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))
            data[f][col_of[oid]] = {"size": [H, W], "counts": rle["counts"].decode()}
    json.dump(data, open(path, "w"))

MASKS = f"{DATA}/masks_original.json"
write_masks(MASKS)

dam = {d["object_id"]: d["description"] for d in json.load(open("data/stage4/sav_000001_first120_descriptions.json"))["objects"]}
pipe = json.load(open("data/stage56/sav_000001_first120_scene_graph_kimi-k3.json"))
pipe_name = {o["object_id"]: o["name"] for o in pipe["objects"]}
OBJ_COLS = sorted(col_of[i] for i in dam)
print(f"{len(columns)} objects after stage 3; giving TraSeR the {len(OBJ_COLS)} DAM described")
print("pipeline label of the floor (id 0):", repr(pipe_name.get(FLOOR)))

## 2. Run TraSeR (released checkpoint, official inference)

`tools/run_traser.py` = their `inference.py` step for step (fp16 because the T4 has no bf16), plus a stats file. Greedy decoding, so a run is repeatable (unlike DAM).

In [ ]:
def run_traser(masks, out):
    t0 = time.time()
    cmd = ["python3", "tools/run_traser.py", "--video", VIDEO, "--masks", masks, "--output", out,
           "--objects", *map(str, OBJ_COLS), "--max_new_tokens", "6000"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"finished in {time.time() - t0:.0f} s, exit code {r.returncode}")
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise SystemExit("TraSeR failed, see the error above")
    stats = json.load(open(out + ".stats.json"))
    for k in ("sampled_frames", "raw_video_tokens", "arranged_tokens", "tokens_dropped_fraction",
              "objects", "window_labels", "generate_s", "json_ok", "out_objects", "out_relationships"):
        print(f"  {k}: {stats.get(k)}")
    return stats

stats1 = run_traser(MASKS, f"{OUT}/traser_original.json")

## 3. Who is who: TraSeR's "object k" → our object id

TraSeR names objects `object 1, 2, …` in the order of the mask columns it kept (objects with no mask on its sampled frames are skipped). The stats file lists those columns.

In [ ]:
import re
def load_graph(path, stats):
    text = open(path).read()
    try:
        g = json.loads(text)
    except json.JSONDecodeError:
        print("output is not complete JSON; reading what parses"); g = {"objects": [], "relationships": []}
        for m in re.finditer(r'\{"object (\d+)": "([^"]*)"', text):
            g["objects"].append({f"object {m.group(1)}": m.group(2), "attributes": []})
    k_to_id = {k + 1: columns[c] for k, c in enumerate(stats["mask_columns"])}
    labels = {}
    for o in g.get("objects", []):
        key = next(k for k in o if k.startswith("object"))
        labels[k_to_id.get(int(key.split()[1]))] = (o[key], o.get("attributes", []))
    rels = []
    for r_ in g.get("relationships", []):
        if len(r_) >= 3:
            rels.append((k_to_id.get(r_[0], r_[0]), r_[1], k_to_id.get(r_[2], r_[2]), r_[3] if len(r_) > 3 else None))
    return labels, rels

lab1, rel1 = load_graph(f"{OUT}/traser_original.json", stats1)
print(f"{'id':>3} | {'pipeline (DAM → LLM)':28} | TraSeR")
for oid in sorted(dam):
    flag = "  <== floor" if oid == FLOOR else "  (boy)" if oid in BOY_PARTS else ""
    print(f"{oid:>3} | {str(pipe_name.get(oid)):28} | {lab1.get(oid, ('—',))[0]}{flag}")
print("\nTraSeR's attributes for the floor:", lab1.get(FLOOR, ("—", []))[1])

In [ ]:
def name(i, labels):
    return "camera" if i == -1 else f"{labels.get(i, ('?',))[0]}({i})"
print(f"TraSeR relations: {len(rel1)} (pipeline: {len(pipe['relationships']['temporal']) + len(pipe['relationships']['spatial'])})")
print("\nTraSeR relations that involve the floor (id 0):")
for s, p, o, span in rel1:
    if FLOOR in (s, o):
        print(f"  {name(s, lab1)} {p} {name(o, lab1)}  {span}")
print("\nPipeline relations that involve the floor (id 0):")
for s, p, o, span, *_ in pipe["relationships"]["temporal"] + pipe["relationships"]["spatial"]:
    if FLOOR in (s, o):
        print(f"  {s} {p} {o}  {span}   ({pipe_name.get(s, s)} {p} {pipe_name.get(o, o)})")

## 4. Run 2: floor mask pulled back from the boy

Same as the DAM experiment, but sized to **TraSeR's** grid: remove every floor pixel within one TraSeR token (28 px of the resized frame) of the boy's outline (convex hull of ids 1, 21, 38). Everything else is unchanged.

In [ ]:
rh, rw = stats1["resized_hw"]
tok_h, tok_w = int(np.ceil(28 * H / rh)), int(np.ceil(28 * W / rw))        # one token in original pixels
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2 * tok_w + 1, 2 * tok_h + 1))
traj = P.ObjectTrajectories(cleaned)
pulled = {}
for f in range(T):
    fl = traj.mask(FLOOR, f)
    if fl is None:
        continue
    parts = [traj.mask(i, f) for i in BOY_PARTS if traj.mask(i, f) is not None]
    hull = np.zeros((H, W), np.uint8)
    if parts:
        cv2.fillConvexPoly(hull, cv2.convexHull(cv2.findNonZero(np.any(parts, axis=0).astype(np.uint8))), 1)
    pulled[f] = fl & ~(cv2.dilate(hull, kernel) > 0)
print(f"one TraSeR token ≈ {tok_w}×{tok_h} original px; floor area kept: "
      f"{np.mean([pulled[f].sum() / max(traj.mask(FLOOR, f).sum(), 1) for f in pulled]):.0%}")
MASKS2 = f"{DATA}/masks_floor_pulled.json"
write_masks(MASKS2, {FLOOR: pulled})
stats2 = run_traser(MASKS2, f"{OUT}/traser_floor_pulled.json")
lab2, rel2 = load_graph(f"{OUT}/traser_floor_pulled.json", stats2)
print("\nfloor label: original mask →", repr(lab1.get(FLOOR, ("—",))[0]), "| pulled-back mask →", repr(lab2.get(FLOOR, ("—",))[0]))

## 5. Save the comparison

In [ ]:
lines = ["# TraSeR vs the SVG2 pipeline on sav_000001 (first 120 frames)", "",
         "Same 40 objects and masks for both. Pipeline = DAM → LLM (lightning) → relations (kimi-k3). "
         "TraSeR = released checkpoint, fp16, greedy.", "",
         f"**Floor (id 0):** pipeline = `{pipe_name.get(FLOOR)}` · TraSeR = `{lab1.get(FLOOR, ('—',))[0]}` · "
         f"TraSeR with floor mask pulled back = `{lab2.get(FLOOR, ('—',))[0]}`", "",
         f"Relations: pipeline {len(pipe['relationships']['temporal']) + len(pipe['relationships']['spatial'])}, "
         f"TraSeR {len(rel1)} (pulled-back run: {len(rel2)})", "",
         "| id | pipeline | TraSeR | TraSeR (floor pulled back) |", "|---|---|---|---|"]
for oid in sorted(dam):
    lines.append(f"| {oid} | {pipe_name.get(oid)} | {lab1.get(oid, ('—',))[0]} | {lab2.get(oid, ('—',))[0]} |")
lines += ["", "## TraSeR relations (original masks)", ""] + [f"- {name(s, lab1)} {p} {name(o, lab1)} {span}" for s, p, o, span in rel1]
lines += ["", "## TraSeR relations (floor pulled back)", ""] + [f"- {name(s, lab2)} {p} {name(o, lab2)} {span}" for s, p, o, span in rel2]
open(f"{OUT}/README.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines[:8]))
print("\nsaved", OUT, "\nthen:  git add", OUT, "notebooks/traser_vs_pipeline.ipynb && git commit -m 'traser vs pipeline' && git push")

**How to read it**

| TraSeR calls the floor… | meaning |
|---|---|
| a person / boy | the model learned the teacher's error: label errors in SVG2 become model errors |
| floor / trampoline / mat | the trained model is more robust than the pipeline that labelled its data |
| person, but floor after pulling back | the model has the same mask-outline weakness as DAM |

Also compare: how many relations each gives, and whether TraSeR relates the shoes to the floor or to the boy.